# A8. TextVQA utility set on Kaggle

Makes the 2,200 images B8 runs: 200 TextVQA questions, each image at a 512 px short side, under the same 11 defence settings as A7. `src/make_textvqa_set.py` picks the questions and reads only those 200 images out of the 7 GB TextVQA zip, so nothing large is downloaded.

**Before you start**
1. Settings: Accelerator **None** (CPU is enough, the OCR step takes about 15 minutes), Internet **on**.
2. Add-ons > Secrets: `GITHUB_TOKEN` switched on.

**How to run**
1. Click **Save Version > Save & Run All (Commit)**. Not Run All, or the files are not kept. About 30 minutes.
2. When it finishes, open the version's **Output** tab. It should list `images_utility.zip`, `ocr_masks_utility.zip`, `textvqa_set.csv` and `utility.csv`.
3. On the Output tab use **New Dataset**, call it `pixelguard-utility`, then **Share** it with Mufeed and Ali.

In [ ]:
!pip install -q easyocr

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient

os.chdir('/kaggle/working')
token = UserSecretsClient().get_secret('GITHUB_TOKEN')
repo = 'github.com/Mehdi-Abidi/VisionLanguageTypographicDefence.git'
subprocess.run(['rm', '-rf', '/tmp/repo'])
r = subprocess.run(['git', 'clone', '-q', f'https://{token}@{repo}', '/tmp/repo'], capture_output=True, text=True)
print('cloned' if r.returncode == 0 else 'clone failed: ' + r.stderr.replace(token, '***'))
subprocess.run(['git', '-C', '/tmp/repo', 'remote', 'set-url', 'origin', f'https://{repo}'])
del token
os.chdir('/tmp/repo')
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

In [ ]:
import time
t0 = time.time()
!python src/make_textvqa_set.py --images --out /tmp/a8
print(f'took {(time.time() - t0) / 60:.0f} min')
same = subprocess.run(['git', 'diff', '--quiet', 'data/textvqa_set.csv', 'manifests/utility.csv']).returncode == 0
print('picked the same 200 questions as the committed CSVs:', same)

In [ ]:
import shutil, zipfile
from pathlib import Path
import pandas as pd

manifest = pd.read_csv('manifests/utility.csv')
missing = [p for p in manifest['path'] if not (Path('/tmp/a8') / p).exists()]
masks = sorted(Path('/tmp/a8/images').glob('*_mask.png'))
with zipfile.ZipFile('/kaggle/working/ocr_masks_utility.zip', 'w') as z:
    for m in masks:
        z.write(m, m.name)
with zipfile.ZipFile('/kaggle/working/images_utility.zip', 'w', zipfile.ZIP_STORED) as z:
    for p in manifest['path']:
        z.write(Path('/tmp/a8') / p, p)
shutil.copy('data/textvqa_set.csv', '/kaggle/working/textvqa_set.csv')
shutil.copy('manifests/utility.csv', '/kaggle/working/utility.csv')
print(len(manifest), 'manifest rows,', len(missing), 'missing,', len(masks), 'OCR masks')
print('output:', {p.name: f'{p.stat().st_size / 1e6:.0f} MB' for p in Path('/kaggle/working').iterdir() if p.is_file()})
print('OK' if not missing and same else 'PROBLEM')